# Projeto Final - ELT537 - Fundamentos de Robótica Aérea

## Missão de Inspeção em Estufa com Drone Autônomo

Imagine uma estufa inteligente vertical utilizada para o cultivo de plantas em ambiente controlado. Nessa estufa, diversas prateleiras organizadas em corredores paralelos abrigam vasos com plantas monitoradas por sensores e câmeras.

<img src="estufa.png" width="70%" align="center">

Seu objetivo é desenvolver o planejamento de voo de um drone quadrotor para realizar inspeções periódicas nos cultivos. O drone deve se deslocar de forma autônoma por entre os corredores, mudar sua **altitude** e se orientar em função da missão dada, de modo que sua câmera fique voltada para plantas específicas.

A missão deve respeitar as seguintes condições:

- A decolagem e o pouso ocorrem em uma **área base** localizada próxima à entrada da estufa;
- O drone deve passar pelas **fileiras selecionadas de cultivo**, voando entre os corredores, mantendo a estabilidade e orientação da câmera;
- É necessário ajustar os **controladores de posição e atitude** para garantir voos suaves e precisos mesmo em ambientes estreitos;
- A orientação do drone em relação ao eixo **global Z** deve ser mantida ou controlada de forma a garantir a captura adequada das imagens.

<br>

A grade abaixo mostra a posição da base de recarga e as posições a serem visitadas. São elas:

- Baia A: (-3m, -0m, 3m, -90°)
- Baia B: (-5m, +3m, 2m, 0°)
- Baia C: (+5m, +2m, 1m, 180°)
- Baia D: (+3m, -5m, 3m, 90°)
- Base de Recarga: (0m, -5m, 1m, 90°)

<img src="grade.png" width="80%" align="center">

Para solucionar o problema proposto, responda as questões a seguir: 

## Questão 1

Quais são os **desafios principais** de controlar um robô aéreo do tipo quadrotor em uma estufa com rotas definidas e orientações alvo?

Os principais desafios são:

- Subatuação: como a translação em x, y ocorre somente inclinando o corpo (φ, θ), posição e orientação ficam acopladas
- Câmera vs. deslocamento: ψ-alvo por baia é independente da direção do voo (ψ precisa de controle próprio)
- Espaço confinado: como os corredores entre as estantes são estreitos, há baixa margem para erro/overshoot
- Aerodinâmica indoor: há efeito solo, recirculação entre prateleiras causam perturbações a rejeitar
- Saturação dos atuadores: com os comandos limitados em [-1, +1], manobras agressivas saturam
- Precisão de altitude: z-alvo muda a cada baia, implica que erro em z pode atingir a prateleira: z-alvo muda a cada baia, implica que erro em z pode atingir a prateleira

## Questão 2

Como você **estruturaria o controle** (por exemplo, separando controle de orientação e de posição) para o problema proposto?

Empregaria o controlador em cascata do AuRoRA para o ArDrone (Inverse Dynamics com Compensador Dinâmico), que separa a malha cinemática (externa) dacompensação dinâmica (interna):
Malha cinemática (externa): a partir do erro de pose X̃ X̃ = X_d − X, gera a velocidade de comando Ucw = Ẋ_d + Ksp·tanh(Kp·X̃ X̃), com o erro de guinada tratadopelo caminho mais curto (|X̃ X̃_ψ| ≤ π)
Compensador dinâmico (interno): Udw = (F·Ku)⁻¹·(U̇ U̇cw + Ksd·(Ucw − Ẋ) + Kv·Ẋ), onde F é a cinemática direta e Ku, Kv vêm do modelo identificado do ArDrone.
Saída (comandos do ARDrone):
Ud = [φ, θ, dZ, dψ], obtidos de Udw e saturados por tanh(·) em (−1, +1). A guinada é um canal próprio, permitindo apontar a câmera independentemente datrajetória x,y,z.

$$X_d, ψ_d -> [ Cinemática: Ucw ] -> [ Compensador: Udw ] -> tanh -> Ud=[φ,θ,dZ,dψ] -> [ ArDrone ] -> X$$

## Questão 3

Como garantir que o drone não colida com as estantes durante a navegação?

Nota: Considere que no problema proposto, o robô não possui implementado estratégias de evasão de obstáculos. A rota a ser seguida deve garantir a navegação sem colisão.

Sem desvio reativo, a segurança é embutida na rota:

- Waypoints no centro dos corredores, com folga mínima (≥ 0,5 m) das estantes
- Trajetória mínimo-jerk, que evita overshoot de posição
- Camadas de altitude: as baias têm z distintos (1 a 3 m). Usa-se altitude para transitar sobre as estantes nas travessias longas
- Rastreamento apertado: o erro de posição observado é pequeno, menor que a folga do corredor

## Questão 4

Quais critérios você usaria para **ajustar os ganhos do controlador** para garantir um comportamento suave, seguro e eficiente?

Usaria os ganhos default do controlador do AuRoRA, organizados por canal [X Y Z
ψ]:



| Matriz | X | Y | Z | ψ |
|--------|---|---|---|---|
| Ksp (proporcional cinemático) | 1,25 | 1,25 | 2 | 1 |
| Ksd (derivativo cinemático) | 1,25 | 1,25 | 2 | 0,5 |
| Kp (saturação tanh) | 1,1 | 1,1 | 1,1 | 1 |
| Kd | 1 | 1 | 1 | 0,1 |


Critérios de sintonia que aplicaria:

- Suavidade: referência mínimo-jerk + feedforward -> posição (x, y, z) com rastreamento colado
- Respeito à saturação: todos os comandos passam por tanh, ficando em [-1, +1]. Nas reorientações rápidas, o canal de guinada atinge o limite (evidência deque a manobra de giro está no limiar do atuador
- Compromisso: ganhos/velocidade da trajetória mais altos -> mais rápido, porém saturação e overshoot de guinada, mais baixos -> suave, porém lento. Alongar os giros (aumentar os tempos de trecho) reduz a saturação de ψ.

## Questão 5

Qual a rota realizada pelo robô para executar a tarefa proposta?

Insira aqui as imagens dos seguintes gráficos:

- Navegação em XYZ
- Evolução temporal de X
- Evolução temporal de Y
- Evolução temporal de Z
- Evolução temporal de $\psi$

<img src="./images/fig-2.png" width="80%" align="center">

<br><br>

<img src="./images/fig-3.png" width="80%" align="center">

<br><br>

<img src="./images/fig-4.png" width="80%" align="center">

<br><br>

<img src="./images/fig-5.png" width="80%" align="center">

<br><br>

<img src="./images/fig-6.png" width="80%" align="center">

## Questão 6

Quais os sinais de controle foram aplicados ao robô para executar a tarefa proposta?

Insira aqui as imagens dos seguintes gráficos:

- Evolução temporal da propulsão vertical
- Evolução temporal dos torques do corpo do veículo

Nota: Os sinais de controle devem ser valores limitados em [-1, +1].

Os comandos do ArDrone são Ud = [φ, θ, dZ, dψ], saturados por tanh e portanto sempre em [-1, +1]. A propulsão vertical corresponde ao comando dZ; os"torques"/atitude do corpo correspondem aos comandos φ (rolagem), θ (arfagem) e dψ (guinada).

<img src="./images/fig-7.png" width="80%" align="center">

<br><br>

<img src="./images/fig-8.png" width="80%" align="center">

-------

In [9]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [11]:
import numpy as np
import matplotlib
# matplotlib.use("Agg")  # deixe comentado para ver inline no Jupyter
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

# ----------------- Parametros (iParameters.m) -----------------
Ts = 1/30
Model_simp = np.array([14.72, 0.2766, 6.233, 0.53, 2.6504, 2.576, 0.3788, 1.5216])
Ku = np.diag(Model_simp[[0, 2, 4, 6]])     # [14.72, 6.233, 2.6504, 0.3788]
Kv = np.diag(Model_simp[[1, 3, 5, 7]])     # [0.2766, 0.53, 2.576, 1.5216]
# Ganhos default do controlador [X Y Z Psi]
gains = np.array([1.25,1.25,2,1, 1.25,1.25,2,0.5, 1.1,1.1,1.1,1, 1,1,1,0.1])
Ksp = np.diag(gains[0:4]); Ksd = np.diag(gains[4:8]); Kp = np.diag(gains[8:12])
# (Kd=gains[12:16] existe no arquivo mas NAO e usado pelo controlador)

# ----------------- Waypoints [x y z psi(graus)] -----------------
WP = np.array([[ 0,-5,0,  90],[ 0,-5,1, 90],[-3, 0,3,-90],[-5, 3,2,  0],
               [ 5, 2,1,180],[ 3,-5,3, 90],[ 0,-5,1, 90],[ 0,-5,0, 90]], float)
segT  = [3,7,5,11,8,4.5,3]
holdT = [0,0,2,2,2,2,0,0]
Pxyz = WP[:,:3]; Ppsi = np.deg2rad(WP[:,3])   # alvos crus (saltos consecutivos <=180 graus)

# ----------------- Trajetoria minimo-jerk -----------------
def minjerk(p0,p1,T,t):
    if T<=0: return np.array(p1,float), np.zeros_like(p1,float)
    s=np.clip(t/T,0,1)
    pos=p0+(p1-p0)*(10*s**3-15*s**4+6*s**5)
    vel=(p1-p0)*(30*s**2-60*s**3+30*s**4)/T
    return pos,vel

segs=[]; tacc=0.0
for i in range(len(WP)-1):
    segs.append((tacc,segT[i],Pxyz[i],Pxyz[i+1],Ppsi[i],Ppsi[i+1])); tacc+=segT[i]
    if holdT[i+1]>0:
        segs.append((tacc,holdT[i+1],Pxyz[i+1],Pxyz[i+1],Ppsi[i+1],Ppsi[i+1])); tacc+=holdT[i+1]
Ttot=tacc

def reference(t):
    for (t0,T,p0,p1,ps0,ps1) in segs:
        if t0<=t<=t0+T+1e-9:
            pos,vel=minjerk(p0,p1,T,t-t0)
            pp,pv=minjerk(np.array([ps0]),np.array([ps1]),T,t-t0)
            return pos,vel,pp[0],pv[0]
    return Pxyz[-1],np.zeros(3),Ppsi[-1],0.0

# ----------------- Controlador (porte exato) -----------------
Ur = np.zeros(4)   # controle cinematico anterior (pSC.Ur)
def controller(X4, dX4, Xd4, dXd4):
    """X4,dX4,Xd4,dXd4 = vetores [x y z psi] / velocidades. Retorna Ud (4)."""
    global Ur
    Xtil = Xd4 - X4
    # wrap do erro de guinada (|Xtil_psi| <= pi)
    if abs(Xtil[3])>np.pi:
        Xtil[3] = (2*np.pi+Xtil[3]) if Xtil[3]<0 else (-2*np.pi+Xtil[3])
    Ucw_ant = Ur.copy()
    Ucw = dXd4 + Ksp@np.tanh(Kp@Xtil)
    dUcw = (Ucw - Ucw_ant)/Ts
    Ur = Ucw.copy()
    psi = X4[3]
    F = np.array([[np.cos(psi),-np.sin(psi),0,0],
                  [np.sin(psi), np.cos(psi),0,0],
                  [0,0,1,0],[0,0,0,1]])
    Udw = np.linalg.solve(F@Ku, dUcw + Ksd@(Ucw - dX4) + Kv@dX4)
    Ud = np.array([-Udw[1], -Udw[0], Udw[2], -Udw[3]])  # [phi theta dZ dpsi]
    return np.tanh(Ud), Udw

# ----------------- Planta [Inferencia] -----------------
def plant_deriv(V, psi, Ud):
    U_ap = np.array([-Ud[1], -Ud[0], Ud[2], -Ud[3]])   # reconstroi Udw aplicado
    F = np.array([[np.cos(psi),-np.sin(psi),0,0],
                  [np.sin(psi), np.cos(psi),0,0],
                  [0,0,1,0],[0,0,0,1]])
    return F@Ku@U_ap - Kv@V     # Vdot no mundo

# ----------------- Simulacao (passo Ts, Euler discreto) -----------------
N=int(np.ceil(Ttot/Ts))+1; t_arr=np.arange(N)*Ts
X = np.array([0,-5,0, np.deg2rad(90)],float)   # [x y z psi]
V = np.zeros(4)                                 # [dx dy dz dpsi]
Xlog=np.zeros((N,4)); Vlog=np.zeros((N,4)); Ulog=np.zeros((N,4)); Rlog=np.zeros((N,4))
for k in range(N):
    t=t_arr[k]
    pos_d,vel_d,psid,dpsid = reference(t)
    Xd4  = np.array([pos_d[0],pos_d[1],pos_d[2],psid])
    dXd4 = np.array([vel_d[0],vel_d[1],vel_d[2],dpsid])
    Ud,_ = controller(X.copy(), V.copy(), Xd4, dXd4)
    Xlog[k]=X; Vlog[k]=V; Ulog[k]=Ud; Rlog[k]=Xd4
    Vdot = plant_deriv(V, X[3], Ud)
    V = V + Ts*Vdot
    X = X + Ts*V
    X[3] = (X[3]+np.pi)%(2*np.pi)-np.pi    # mantem psi em [-pi,pi] (como a plataforma)

err=Xlog[:,:3]-Rlog[:,:3]; rmse=np.sqrt(np.mean(np.sum(err**2,axis=1)))
print(f"Tempo total: {Ttot:.1f} s | RMSE posicao: {rmse*100:.2f} cm")
for i,nm in enumerate(["phi","theta","dZ","dpsi"]):
    print(f"  |Ud_{nm}| max = {np.max(np.abs(Ulog[:,i])):.3f}")

# ----------------- Figuras (mesmo layout do MATLAB) -----------------
plt.rcParams.update({'font.size':12,'axes.grid':True,'grid.alpha':0.3,'figure.dpi':120})
navy=(0.12,0.31,0.47); red=(0.75,0.22,0.17); grn=(0.18,0.49,0.20); org=(0.88,0.54,0)
tv=t_arr
x,y,z,psi = Xlog[:,0],Xlog[:,1],Xlog[:,2],np.rad2deg(Xlog[:,3])
xr,yr,zr,psir = Rlog[:,0],Rlog[:,1],Rlog[:,2],np.rad2deg(Rlog[:,3])

fig=plt.figure(figsize=(9,6)); ax=fig.add_subplot(111,projection='3d')
ax.plot(x,y,z,color=navy,lw=2)
for i in range(2,6): ax.scatter(WP[i,0],WP[i,1],WP[i,2],color=red,s=60,depthshade=False)
ax.set_xlabel('X [m]');ax.set_ylabel('Y [m]');ax.set_zlabel('Z [m]')
ax.set_title('Navegacao 3D (XYZ)'); ax.view_init(22,-60)
plt.tight_layout(); plt.savefig('images/ap_fig2_xyz.png',bbox_inches='tight'); plt.show()

def ts(f,ref,act,ylab,ttl):
    plt.figure(figsize=(9,3.6))
    plt.plot(tv,ref,'--',color=red,lw=2,label='Referencia')
    plt.plot(tv,act,color=navy,lw=1.8,label='Real')
    plt.xlabel('Tempo [s]');plt.ylabel(ylab);plt.title(ttl);plt.legend(loc='best')
    plt.tight_layout();plt.savefig(f,bbox_inches='tight');plt.show()
ts('images/ap_fig3_x.png',xr,x,'X [m]','Evolucao temporal de X')
ts('images/ap_fig4_y.png',yr,y,'Y [m]','Evolucao temporal de Y')
ts('images/ap_fig5_z.png',zr,z,'Z [m]','Evolucao temporal de Z')
ts('images/ap_fig6_psi.png',psir,psi,r'$\psi$ [graus]','Evolucao temporal de $\\psi$')

plt.figure(figsize=(9,3.6)); 
for lv in (1,-1): plt.axhline(lv,ls=':',color=(.6,.6,.6))
plt.plot(tv,Ulog[:,2],color=grn,lw=2,label='u_z (dZ)')
plt.ylim(-1.05,1.05);plt.xlabel('Tempo [s]');plt.ylabel('Sinal normalizado')
plt.title('Sinal de controle - Propulsao vertical');plt.legend(loc='upper right')
plt.tight_layout();plt.savefig('images/ap_fig7_uz.png',bbox_inches='tight');plt.show()

plt.figure(figsize=(9,3.8))
for lv in (1,-1): plt.axhline(lv,ls=':',color=(.6,.6,.6))
plt.plot(tv,Ulog[:,0],color=red ,lw=1.6,label=r'$u_\phi$ (rolagem)')
plt.plot(tv,Ulog[:,1],color=navy,lw=1.6,label=r'$u_\theta$ (arfagem)')
plt.plot(tv,Ulog[:,3],color=org ,lw=1.6,label=r'$u_\psi$ (guinada)')
plt.ylim(-1.05,1.05);plt.xlabel('Tempo [s]');plt.ylabel('Sinal normalizado')
plt.title('Sinais de controle - Comandos de atitude (corpo)');plt.legend(loc='upper right',ncol=3)
plt.tight_layout();plt.savefig('images/ap_fig8_torques.png',bbox_inches='tight');plt.show()
print("figuras geradas")

Tempo total: 49.5 s | RMSE posicao: 2.19 cm
  |Ud_phi| max = 0.185
  |Ud_theta| max = 0.068
  |Ud_dZ| max = 0.767
  |Ud_dpsi| max = 0.999


/var/folders/45/4lf3s2612vjg4ydsqs4rcxwh0000gn/T/ipykernel_19513/2848813805.py:110: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('images/ap_fig2_xyz.png',bbox_inches='tight'); plt.show()
/var/folders/45/4lf3s2612vjg4ydsqs4rcxwh0000gn/T/ipykernel_19513/2848813805.py:117: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig(f,bbox_inches='tight');plt.show()


figuras geradas


/var/folders/45/4lf3s2612vjg4ydsqs4rcxwh0000gn/T/ipykernel_19513/2848813805.py:128: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig('images/ap_fig7_uz.png',bbox_inches='tight');plt.show()
/var/folders/45/4lf3s2612vjg4ydsqs4rcxwh0000gn/T/ipykernel_19513/2848813805.py:137: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig('images/ap_fig8_torques.png',bbox_inches='tight');plt.show()
